# Tutorial 130 — Chat with a QAChat project

The earlier tutorials built the pieces separately. This tutorial combines them
into the normal QAChat project workflow: source synchronization, reusable
indexes, embedding-based and deterministic retrieval, a chat model, citations,
conversation persistence, and optional tracing.

You will choose either a code or literature project, inspect the exact backend
and model pair selected from `profile.ini`, open or resume a `RAGSession`, ask
follow-up questions, inspect the evidence supplied to the LLM, and locate the
persisted project state.

[Tutorial 140](./140_logging_trace.ipynb) uses the same workflow but follows one
execution through the trace in detail.


## 1. Prepare QAChat

Each tutorial is independently runnable. These cells locate the repository,
make the local package importable, and call `prepare_notebook()` to verify the
project `.venv` environment. If Jupyter asks you to select the printed `.venv`
kernel, select it and rerun the cell.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


In [ ]:
import qachat


## 2. Choose the project

The notebook can use either a bundled example or another local source folder.
Normally these are the only settings you need to edit:

- `PROJECT_KIND` selects a code or literature project;
- `PROJECT_DIRECTORY` selects the original source folder;
- `PROJECT_DIRECTORY = None` uses the corresponding bundled example;
- `ENABLE_TRACE` controls whether the session records a forensic trace.

QAChat exposes canonical project-kind constants, but the underlying values remain
ordinary strings. The original source directory is treated as input. QAChat
writes its managed state to a separate workspace under `sandbox/`.


In [ ]:
PROJECT_KIND = qachat.PROJECT_KIND_LITERATURE
PROJECT_DIRECTORY = None  # "/path/to/my/source-project"
ENABLE_TRACE = True


In [ ]:
if PROJECT_KIND not in qachat.PROJECT_KIND_CHOICES:
    raise ValueError(
        f"PROJECT_KIND must be one of {qachat.PROJECT_KIND_CHOICES}."
    )


In [ ]:
if PROJECT_DIRECTORY is None:
    example_name = (
        "code_repo"
        if PROJECT_KIND == qachat.PROJECT_KIND_CODE
        else "literature_repo"
    )
    source_directory = (
        root / "example_repos" / example_name
    ).resolve(strict=True)
    print(f"Included example source: {source_directory}")
else:
    source_directory = (
        Path(PROJECT_DIRECTORY).expanduser().resolve(strict=True)
    )
    print(f"Original source: {source_directory}")


QAChat never writes project settings or indexes into the original source
folder. It mirrors the selected files into a managed workspace under `sandbox/`.
For example, `example_repos/code_repo` becomes
`sandbox/project_code_repo/code_repo`, with `indexes/`, `logs/`, `profile.ini`,
and `source_snapshot.json` beside the copied source folder.


## 3. Open or resume QAChat

`prepare_project_workspace()` compares source manifests and synchronizes changed
files into the managed workspace. `RAGSession` then checks whether the persistent
retrieval index still matches both the copied source and the indexing settings.
A current index is reused; a stale or missing index is rebuilt.

The same workflow is used for bundled examples and user-provided projects. The
original source folder is never used as QAChat's state directory.


The following cell prints the managed workspace paths so you can see where the
copied source, profile, indexes, logs, and source snapshot live before any model
is loaded.


In [ ]:
project = qachat.prepare_project_workspace(
    source_directory,
    PROJECT_KIND,
    root / "sandbox",
)

project_directory = project.root
profile_path = project.root / "profile.ini"

print(f"QAChat workspace: {project.root}")
print(f"Indexed source snapshot: {project.source}")
print(f"Project settings: {profile_path}")


### Resolve the models that QAChat will load

`RAGSession` reads model choices from `profile.ini`. The profile stores separate
chat and embedding model IDs for MLX and PyTorch. QAChat resolves the requested
backend first and then selects the corresponding model pair.

Showing that automatic resolution explicitly here makes the runtime choice easy
to audit. To use different models for this project, edit `profile.ini` and
reopen the project.


In [ ]:
SELECTED_BACKEND = qachat.choose_backend(project.profile.backend)
EMBEDDING_MODEL = project.profile.embedding_model(SELECTED_BACKEND)
CHAT_MODEL = project.profile.chat_model(SELECTED_BACKEND)

print(f"Requested backend: {project.profile.backend}")
print(f"Resolved backend: {SELECTED_BACKEND}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Chat model: {CHAT_MODEL}")


In [ ]:
index_path = project.indexes / "retrieval.json"


### Prepare progress reporting

Import the notebook display tools and the QAChat progress API. If a previous
QAChat session exists, close it before opening the project again.


In [ ]:
from html import escape
from IPython.display import HTML, display


if "session" in globals():
    session.close()


### Create the notebook progress display

The display is updated in place instead of printing a new line for every progress
event. `format_duration()` converts seconds into readable times such as `2m 15s`.


In [ ]:
progress_display = display(HTML(""), display_id=True)


def format_duration(seconds: float | None) -> str:
    if seconds is None:
        return "--"

    seconds = max(0, round(seconds))
    minutes, seconds = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)

    if hours:
        return f"{hours}h {minutes:02d}m {seconds:02d}s"
    if minutes:
        return f"{minutes}m {seconds:02d}s"
    return f"{seconds}s"


### Define how progress is shown

QAChat sends `ProgressEvent` objects while preparing the project. This callback
turns those events into a live progress bar.

**Important:** `Stage ETA` estimates the remaining time for the **current stage only**
(for example, chunking or embedding). It is **not** an estimate of the remaining time
for the complete indexing workflow. The percentage is the overall workflow progress.


In [ ]:
def show_progress(event: qachat.ProgressEvent) -> None:
    width = 28
    percent = max(0.0, min(100.0, event.percent))
    filled = round(width * percent / 100.0)
    bar = "█" * filled + "░" * (width - filled)

    details = []
    if event.current is not None and event.total is not None:
        details.append(f"{event.current:,}/{event.total:,}")
    if event.remaining is not None:
        details.append(f"{event.remaining:,} remaining")
    if event.elapsed_seconds is not None:
        details.append(f"elapsed {format_duration(event.elapsed_seconds)}")
    if event.eta_seconds is not None:
        details.append(f"Stage ETA {format_duration(event.eta_seconds)}")

    detail_text = "  |  ".join(details)
    lines = [
        f"[{bar}] {percent:5.1f}%  {event.message}",
    ]
    if detail_text:
        lines.append(detail_text)

    progress_display.update(
        HTML(f"<pre style='margin:0'>{escape(chr(10).join(lines))}</pre>")
    )


### Open the QAChat session

This is the cell that actually checks the saved index and prepares QAChat. If the
project is unchanged, the existing index is reused. If indexed files or indexing
settings changed, QAChat rebuilds the index and reports progress through the callback
above. With `ENABLE_TRACE = True`, the session also records exact prompts, token
inputs, input/output token counts, retrieval decisions, per-call timings, settings,
and failures in `logs/traces`.


In [ ]:
session = qachat.RAGSession(
    project.root,
    progress_callback=show_progress,
    trace=qachat.TraceConfig(
        enabled=ENABLE_TRACE,
        mode=qachat.TRACE_MODE_FORENSIC,
    ),
)
if session.trace.enabled:
    print(f"Readable trace: {session.trace.readable_path}")


In [ ]:
if session.turns:
    print("Resumed a previous conversation.")
    print(f"Restored turns: {len(session.turns)}")
    print(f"Last question: {session.turns[-1].question}")
elif index_path.is_file() and session.index_status == "reused":
    print("Opened an existing project and reused its index.")
else:
    print("Opened the project and built its index.")

changes = session.source_changes
print(f"Index status: {session.index_status}")
print(f"Backend: {session.backend}")
print(f"Source snapshot: {session.project.source}")

if session.index_status == "rebuilt":
    print(
        "Source changes: "
        f"+{len(changes['added'])} "
        f"~{len(changes['modified'])} "
        f"-{len(changes['removed'])}"
    )
    if changes["configuration_changed"]:
        print("Index settings are new or changed.")


## 4. Ask a question

Choose or edit `QUESTION`, then call `session.ask()`. One call runs retrieval,
constructs the evidence-grounded prompt, invokes the chat model, and immediately
persists the question, answer, evidence, and effective settings for that turn.


In [ ]:
DEFAULT_QUESTIONS = {
    qachat.PROJECT_KIND_CODE: (
        "How does this project convert Celsius to Kelvin?"
    ),
    qachat.PROJECT_KIND_LITERATURE: (
        "What did the toy sleep study observe?"
    ),
}

QUESTION = DEFAULT_QUESTIONS[session.project.profile.kind]

print(f"Question: {QUESTION}")


In [ ]:
answer = session.ask(QUESTION)

print(f"Answer: {answer}")


## 5. Inspect the evidence

The latest conversation turn retains the exact source chunks supplied to the
LLM. Their citation numbers correspond to the references requested in the model
answer, so you can audit which repository text supported the response.


In [ ]:
latest_turn = session.turns[-1]

for item in latest_turn.evidence:
    location = f"{item.path}:{item.start_line}-{item.end_line}"
    print(f"[{item.citation}] {location}")
    print(item.text)
    print()


## 6. Keep asking questions

Keep the same `RAGSession` open for follow-up questions. The next turn can use
earlier user and assistant messages as conversation history while performing a
fresh retrieval for the new question.


In [ ]:
NEXT_QUESTION = "Can you explain that in more detail?"

print(f"Next question: {NEXT_QUESTION}")


In [ ]:
next_answer = session.ask(NEXT_QUESTION)
latest_turn = session.turns[-1]

print(f"Answer: {next_answer}")


In [ ]:
print("Evidence:")
for item in latest_turn.evidence:
    location = f"{item.path}:{item.start_line}-{item.end_line}"
    print(f"[{item.citation}] {location}")


## 7. Inspect persisted project state

The managed project keeps configuration, conversation state, retrieval indexes,
and optional traces separate from the original source. The following cells show
where the reusable `profile.ini`, active `conversation.json`, and index files are
stored.


In [ ]:
profile_path = session.project.root / "profile.ini"

print(f"Project settings: {profile_path}")


**conversation.json** contains every question, answer, evidence set, and
parameter snapshot in the active conversation.


In [ ]:
conversation_path = session.project.logs / "conversation.json"

print(f"Active conversation: {conversation_path}")


In [ ]:
print(f"Reusable indexes: {session.project.indexes}")


In [ ]:
import json


print(json.dumps(latest_turn.parameters, indent=2, sort_keys=True))


## 8. Optionally start a new conversation

Starting a new conversation is deliberately opt-in. Leave
`START_NEW_CONVERSATION = False` during ordinary tutorial runs. If you set it to
`True`, QAChat archives the active conversation log with a UTC timestamp, clears
conversation history from memory, and retains the reusable retrieval indexes.


In [ ]:
START_NEW_CONVERSATION = False

if START_NEW_CONVERSATION:
    archived_path = session.reset()
    print(f"Archived conversation: {archived_path}")
    print(f"Active turns: {len(session.turns)}")


## 9. Close QAChat

Closing releases the models without removing indexes or conversations.

In [ ]:
session.close()

print("QAChat closed safely.")


## Continue the tutorial series

**Previous:** [Tutorial 120 — Expand retrieval deterministically](./120_retrieval_routing.ipynb)

**Next:** [Tutorial 140 — Inspect a QAChat execution trace](./140_logging_trace.ipynb)
